Use an LLM to generate human-style risk flags and recommendations from unstructured text, and apply them at the decision policy layer, not inside the predictive model.

In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [4]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv("../data/processed/credit_tab_text_conditioned.csv")

xgb = joblib.load("../src/models/xgb_cost_sensitive.pkl")

In [5]:
# X = df[[
#     "age",
#     "income",
#     "debt_to_income",
#     "credit_score",
#     "loan_amount"
# ]]

# proba = xgb.predict_proba(X)[:, 1]
# df["model_proba"] = proba

In [6]:
import joblib

# Load the exact test indices used in Notebook 08
test_idx = joblib.load("../data/processed/test_indices.pkl")

# Subset ONLY the test data
df_test = df.loc[test_idx].copy()

X_test = df_test[[
    "age",
    "income",
    "debt_to_income",
    "credit_score",
    "loan_amount"
]]

# Predict probabilities on test set only
df_test["model_proba"] = xgb.predict_proba(X_test)[:, 1]

In [7]:
def build_prompt(row):
    return f"""
    Loan officer notes: {row['officer_notes']}
    Location description: {row['location_text']}

    Based on this information, classify the borrower into:
    - LOW_RISK
    - MEDIUM_RISK
    - HIGH_RISK

    Output only the category.
    """


In [8]:
def simulated_llm_risk(row):
    text = row["officer_notes"].lower()

    if "high debt" in text or "past credit issues" in text:
        return "HIGH_RISK"
    elif "volatile" in text or "irregular" in text:
        return "MEDIUM_RISK"
    else:
        return "LOW_RISK"

df_test["llm_risk_flag"] = df_test.apply(simulated_llm_risk, axis=1)

In [9]:
def policy_decision_soft(row):
    p = row["model_proba"]
    flag = row["llm_risk_flag"]

    # Guardrail only when model is uncertain
    if flag == "HIGH_RISK" and p < 0.75:
        return 0

    if flag == "MEDIUM_RISK" and p < 0.6:
        return 0

    # Otherwise follow model
    return int(p >= 0.5)

In [10]:
df_test["policy_decision"] = df_test.apply(policy_decision_soft, axis=1)
len(df_test)

3000

In [12]:
df_test.columns



Index(['age', 'income', 'debt_to_income', 'credit_score', 'loan_amount',
       'location_text', 'officer_notes', 'approved', 'model_proba',
       'llm_risk_flag', 'policy_decision'],
      dtype='object')

In [13]:
from sklearn.metrics import confusion_matrix

y_true = df_test["approved"]
y_pred_policy = df_test["policy_decision"]

cm = confusion_matrix(y_true, y_pred_policy)
TN, FP, FN, TP = cm.ravel()

cost_fp = 100
cost_fn = 10

expected_cost = FP * cost_fp + FN * cost_fn
cost_per_applicant = expected_cost / len(df_test)

cm, expected_cost, cost_per_applicant

(array([[1166,   34],
        [ 481, 1319]]),
 8210,
 2.736666666666667)

The model did not improve prediction. Also, the model is running on the entire set of 10,000 rows instead of a training set of 3,000 rows as in Notebook 8. Therefore, we cannot do apples-to-apples comparison. 